# Option B: BC-Warmstarted FCP-style PPO (v4)

**Architecture**: MLP(512→256→128) + obs stack K=4 + BC warm-start + FCP-style PPO

**Key corrections (GUIA_CORRECCIONES_OPTION_A2)**:
- prev_action[t] = action[t-1], BOS=0 for t=0
- Quality tiers, checkpoint by soups, PyTorch→NumPy parity verified


In [ ]:
import os, sys, json, csv, time, pathlib, shutil, traceback, subprocess, zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import deque
from torch.utils.data import Dataset, DataLoader

OUTPUT_DIR = pathlib.Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

progress = {'status': 'running', 'stage': 'init', 'artifacts': []}
def save_progress():
    (OUTPUT_DIR / 'run_summary.json').write_text(json.dumps(progress, indent=2), encoding='utf-8')
save_progress()

print(f'GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'  {torch.cuda.get_device_name(0)}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


In [ ]:
progress['stage'] = 'install'; save_progress()

def run_cmd(cmds, quiet=True):
    r = subprocess.run(cmds, capture_output=True, text=True)
    if not quiet or r.returncode != 0:
        if r.stdout: print('  OUT:', r.stdout[-300:])
        if r.stderr: print('  ERR:', r.stderr[-300:])
    return r.returncode == 0

OVERCOOKED_INSTALLED = False

# Attempt 1: PyPI install (relaxed dependencies)
print('[1] Trying PyPI install of overcooked-ai...')
if run_cmd(['pip', 'install', 'overcooked-ai', '--no-deps', '-q']):
    run_cmd(['pip', 'install', 'PyYAML>=6.0', 'scipy', 'dill', 'flask', '-q'])
    try:
        import numpy as np
        np.Inf = np.inf
        import overcooked_ai_py; OVERCOOKED_INSTALLED = True; print('  OK')
    except Exception as e: print(f'  Import failed: {e}')

# Attempt 2: GitHub source
if not OVERCOOKED_INSTALLED:
    print('[2] Trying GitHub source...')
    if run_cmd(['pip', 'install', 'git+https://github.com/HumanCompatibleAI/overcooked_ai.git', '--no-deps', '-q']):
        run_cmd(['pip', 'install', 'PyYAML>=6.0', 'scipy', 'dill', '-q'])
        try:
            import numpy as np
            np.Inf = np.inf
            import overcooked_ai_py; OVERCOOKED_INSTALLED = True; print('  OK')
        except Exception as e: print(f'  Import failed: {e}')

print(f'OVERCOOKED_INSTALLED: {OVERCOOKED_INSTALLED}')
progress['overcooked_available'] = OVERCOOKED_INSTALLED; save_progress()


In [ ]:
progress['stage'] = 'validate_inputs'; save_progress()

# Extract any zip files in /kaggle/input
for zip_path in pathlib.Path('/kaggle/input').rglob('*.zip'):
    print(f'Extracting dataset zip: {zip_path}...')
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('/kaggle/working')
        print('  Extracted OK')
    except Exception as e:
        print(f'  Failed to extract {zip_path}: {e}')

# Look for npz files in /kaggle/working/data or /kaggle/input/
DATA_DIR = None
npz_files = []

for p in [pathlib.Path('/kaggle/working/data'), pathlib.Path('/kaggle/working')] + list(pathlib.Path('/kaggle/input').glob('*')):
    if p.exists() and p.is_dir():
        found = list(p.rglob('*.npz'))
        print(f'  {p}: {len(found)} npz files')
        if len(found) > len(npz_files):
            DATA_DIR = p
            npz_files = found

print(f'Using data from: {DATA_DIR}')
print(f'Total npz files: {len(npz_files)}')

if not npz_files:
    # create synthetic minimal dataset for pipeline testing
    print('WARNING: No data found. Creating minimal synthetic dataset for testing...')
    DATA_DIR = OUTPUT_DIR / 'synthetic_data'
    DATA_DIR.mkdir(exist_ok=True)
    OBS_DIM = 96
    for i in range(20):
        T = 250
        obs = np.random.randn(T, OBS_DIM).astype(np.float32)
        actions = np.random.randint(0, 6, T).astype(np.int64)
        actions[50] = 5; actions[100] = 5; actions[150] = 5
        rewards = np.zeros(T, dtype=np.float32)
        rewards[60] = 20.0; rewards[110] = 20.0
        dones = np.zeros(T, dtype=bool); dones[-1] = True
        ep_ids = np.zeros(T, dtype=int)
        agent_idxs = np.full(T, i % 2, dtype=int)
        np.savez_compressed(DATA_DIR / f'synthetic_ep_{i:03d}.npz',
            obs=obs, actions=actions, rewards=rewards, dones=dones,
            episode_ids=ep_ids, agent_indices=agent_idxs)
    npz_files = list(DATA_DIR.rglob('*.npz'))
    print(f'Created {len(npz_files)} synthetic episodes')
    progress['synthetic_data'] = True; save_progress()

# Validate first file
f = npz_files[0]; d = np.load(f, allow_pickle=True)
print(f'Sample: {f.name}, keys={list(d.keys())}')
print(f'obs shape: {d["obs"].shape}')
OBS_DIM = d['obs'].shape[1] if d['obs'].ndim > 1 else 96
print(f'OBS_DIM = {OBS_DIM}')


In [ ]:
class MLP(nn.Module):
    def __init__(self, layer_sizes, activation='relu', use_layer_norm=True, dropout=0.0):
        super().__init__()
        layers = []
        for i in range(len(layer_sizes) - 1):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
            is_last = (i == len(layer_sizes) - 2)
            if not is_last:
                if use_layer_norm: layers.append(nn.LayerNorm(layer_sizes[i+1]))
                layers.append(nn.ReLU())
                if dropout > 0: layers.append(nn.Dropout(dropout))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)


class BCWarmstartActor(nn.Module):
    def __init__(self, obs_dim=96, k_stack=4, num_actions=6, hidden_sizes=(512,256,128), dropout=0.1):
        super().__init__()
        self.obs_dim=obs_dim; self.k_stack=k_stack; self.num_actions=num_actions
        input_dim = k_stack*obs_dim + 2 + num_actions
        self.encoder = MLP([input_dim]+list(hidden_sizes), dropout=dropout)
        self.actor_head = nn.Linear(hidden_sizes[-1], num_actions)
        for m in self.modules():
            if isinstance(m, nn.Linear): nn.init.orthogonal_(m.weight); nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.actor_head.weight, gain=0.01)

    def forward(self, stack_obs, agent_index, prev_action):
        ah = F.one_hot(agent_index.long(), 2).float()
        ph = F.one_hot(prev_action.long(), self.num_actions).float()
        return self.actor_head(self.encoder(torch.cat([stack_obs, ah, ph], -1)))


class ActorCritic(nn.Module):
    def __init__(self, obs_dim=96, k_stack=4, num_actions=6, hidden_sizes=(512,256,128), dropout=0.05):
        super().__init__()
        self.obs_dim=obs_dim; self.k_stack=k_stack; self.num_actions=num_actions
        input_dim = k_stack*obs_dim + 2 + num_actions
        ls = [input_dim]+list(hidden_sizes)
        self.actor_encoder = MLP(ls, dropout=dropout)
        self.actor_head = nn.Linear(hidden_sizes[-1], num_actions)
        self.critic_encoder = MLP(ls, dropout=dropout)
        self.critic_head = nn.Linear(hidden_sizes[-1], 1)
        for m in self.modules():
            if isinstance(m, nn.Linear): nn.init.orthogonal_(m.weight); nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.actor_head.weight, gain=0.01)

    def _inp(self, so, ai, pa):
        return torch.cat([so, F.one_hot(ai.long(),2).float(), F.one_hot(pa.long(),self.num_actions).float()], -1)

    def actor_logits(self, so, ai, pa):
        return self.actor_head(self.actor_encoder(self._inp(so, ai, pa)))

    def forward(self, so, ai, pa):
        x = self._inp(so, ai, pa)
        return self.actor_head(self.actor_encoder(x)), self.critic_head(self.critic_encoder(x)).squeeze(-1)

    def load_from_bc(self, bc):
        self.actor_encoder.load_state_dict(bc.encoder.state_dict())
        self.actor_head.load_state_dict(bc.actor_head.state_dict())

print('Models OK')


In [ ]:
progress['stage'] = 'dataset'; save_progress()

def quality_tier(rew, act):
    s = float(rew.sum()) / 20.0
    if s >= 2: return 'A', 1.5
    if s >= 1: return 'B', 1.0
    stay = (act==4).mean()
    mx = 1; c = 1
    for i in range(1, len(act)):
        c = c+1 if act[i]==act[i-1] else 1; mx = max(mx, c)
    if (act==5).sum()>5 and stay<0.8 and mx<50: return 'C', 0.4
    if stay>0.9 or mx>100: return 'D', 0.0
    return 'C', 0.4

def load_episodes(data_dir):
    eps = []; seen = set()
    for fp in sorted(pathlib.Path(data_dir).rglob('*.npz')):
        if fp.stem in seen: continue
        seen.add(fp.stem)
        try:
            d = np.load(fp, allow_pickle=True)
            obs=d['obs'].astype(np.float32); acts=d['actions'].astype(np.int64)
            rews=d.get('rewards', np.zeros(len(acts), dtype=np.float32))
            epids=d.get('episode_ids', np.zeros(len(obs),int))
            agidxs=d.get('agent_indices', np.zeros(len(obs),int))
            for eid in np.unique(epids):
                m=epids==eid
                if m.sum()<5: continue
                tier,w = quality_tier(rews[m], acts[m])
                if tier=='D': continue
                eps.append({'obs':obs[m],'actions':acts[m],'rewards':rews[m],
                            'agent_index':int(agidxs[m][0]),'quality_tier':tier,
                            'quality_weight':w,'soups':float(rews[m].sum())/20})
        except Exception as e:
            print(f'Error reading {fp}: {e}')
    return eps

episodes = load_episodes(DATA_DIR)
print(f'Loaded {len(episodes)} episodes')
tc = {}; [tc.update({e['quality_tier']: tc.get(e['quality_tier'],0)+1}) for e in episodes]
print('Tiers:', tc, '| mean_soups:', round(np.mean([e['soups'] for e in episodes]),2))

# Split by episode
rng = np.random.default_rng(42)
idx = rng.permutation(len(episodes)).tolist()
n_tr = int(len(idx)*.80); n_v = int(len(idx)*.10)
train_eps = [episodes[i] for i in idx[:n_tr]]
val_eps = [episodes[i] for i in idx[n_tr:n_tr+n_v]]

# Normalization from train only
all_tr = np.concatenate([e['obs'] for e in train_eps], 0)
NORM_MEAN = all_tr.mean(0).astype(np.float32)
NORM_STD = np.maximum(all_tr.std(0), 1e-8).astype(np.float32)
norm_stats = {'mean': NORM_MEAN.tolist(), 'std': NORM_STD.tolist()}
(OUTPUT_DIR/'normalization.json').write_text(json.dumps(norm_stats, indent=2))
print(f'Normalization computed. obs_dim={len(NORM_MEAN)}')

progress['artifacts'].append('normalization.json'); save_progress()


In [ ]:
K_STACK=4; BATCH=512

class BCDataset(Dataset):
    def __init__(self, eps, nm, ns, k=4):
        self.k=k; self.smp=[]
        all_a = np.concatenate([e['actions'] for e in eps])
        cnt = np.maximum(np.bincount(all_a, minlength=6).astype(float), 1.)
        aw = np.clip(cnt.sum()/(6*cnt)/cnt.sum()*cnt.sum(), .5, 3.).astype(np.float32)
        for ep in eps:
            on = (ep['obs']-nm)/ns; acts=ep['actions']; ai=ep['agent_index']; qw=ep['quality_weight']
            for t in range(len(acts)):
                st = np.concatenate([on[max(0,t-j)] for j in range(k-1,-1,-1)]).astype(np.float32)
                pa = int(acts[t-1]) if t>0 else 0
                ta = int(acts[t])
                w = float(np.clip(qw*aw[ta], .05, 5.))
                self.smp.append((st, ai, pa, ta, w))
    def __len__(self): return len(self.smp)
    def __getitem__(self, i):
        st,ai,pa,ta,w = self.smp[i]
        return torch.from_numpy(st), torch.tensor(ai,dtype=torch.long), torch.tensor(pa,dtype=torch.long), torch.tensor(ta,dtype=torch.long), torch.tensor(w,dtype=torch.float32)

print('Building datasets...')
tr_ds = BCDataset(train_eps, NORM_MEAN, NORM_STD, K_STACK)
vl_ds = BCDataset(val_eps, NORM_MEAN, NORM_STD, K_STACK)
print(f'Train: {len(tr_ds)}, Val: {len(vl_ds)} samples')
tr_dl = DataLoader(tr_ds, BATCH, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
vl_dl = DataLoader(vl_ds, BATCH, shuffle=False, num_workers=2)


In [ ]:
progress['stage']='bc_warmstart'; save_progress()

HS=(512,256,128)
bc_model = BCWarmstartActor(OBS_DIM, K_STACK, 6, HS, dropout=0.1).to(DEVICE)
print(f'BC params: {sum(p.numel() for p in bc_model.parameters()):,}')

opt_bc = optim.Adam(bc_model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt_bc, T_max=60, eta_min=1e-5)

best_vl = float('inf'); pat=0; BC_PAT=12; history=[]

for epoch in range(1, 61):
    t0=time.time()
    bc_model.train()
    tl=tc_=tt_=0
    for st,ai,pa,ta,sw in tr_dl:
        st,ai,pa,ta,sw = st.to(DEVICE),ai.to(DEVICE),pa.to(DEVICE),ta.to(DEVICE),sw.to(DEVICE)
        logits=bc_model(st,ai,pa)
        ce=F.cross_entropy(logits,ta,reduction='none',label_smoothing=0.02)
        loss=(ce*sw).sum()/sw.sum().clamp(1.)
        opt_bc.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(bc_model.parameters(), 1.); opt_bc.step()
        tl+=loss.item()*len(ta); tc_+=(logits.argmax(-1)==ta).sum().item(); tt_+=len(ta)
    sched.step()

    bc_model.eval(); vl=vc=vt=0
    with torch.no_grad():
        for st,ai,pa,ta,sw in vl_dl:
            st,ai,pa,ta,sw=st.to(DEVICE),ai.to(DEVICE),pa.to(DEVICE),ta.to(DEVICE),sw.to(DEVICE)
            logits=bc_model(st,ai,pa)
            ce=F.cross_entropy(logits,ta,reduction='none')
            l=(ce*sw).sum()/sw.sum().clamp(1.)
            vl+=l.item()*len(ta); vc+=(logits.argmax(-1)==ta).sum().item(); vt+=len(ta)

    tl_=tl/max(tt_,1); vl_=vl/max(vt,1); el=time.time()-t0
    row={'epoch':epoch,'train_loss':round(tl_,6),'train_acc':round(tc_/max(tt_,1),4),
         'val_loss':round(vl_,6),'val_acc':round(vc/max(vt,1),4),'elapsed_s':round(el,2)}
    history.append(row)
    print(f'  E{epoch:3d}: tr={tl_:.4f} vl={vl_:.4f} va={vc/max(vt,1):.3f} ({el:.1f}s)')

    if vl_ < best_vl:
        best_vl=vl_; pat=0
        torch.save({'epoch':epoch,'model_state_dict':bc_model.state_dict(),'val_loss':vl_,
                    'obs_dim':OBS_DIM,'k_stack':K_STACK,'hidden_sizes':list(HS)}, OUTPUT_DIR/'bc_warmstart.pt')
        print(f'    ✓ best val_loss={best_vl:.4f}')
        progress['bc_best_val_loss']=best_vl; save_progress()
    else:
        pat+=1
        if pat>=BC_PAT: print(f'Early stop at {epoch}'); break

with open(OUTPUT_DIR/'option_b_bc_training.csv','w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(history[0].keys())); w.writeheader(); w.writerows(history)
progress['artifacts'].extend(['bc_warmstart.pt','option_b_bc_training.csv']); save_progress()
print(f'BC done. best_val_loss={best_vl:.6f}')


In [ ]:
ck = torch.load(OUTPUT_DIR/'bc_warmstart.pt', map_location=DEVICE, weights_only=False)
bc_model.load_state_dict(ck['model_state_dict']); bc_model.eval()
print(f'Loaded BC epoch={ck["epoch"]}, val_loss={ck["val_loss"]:.4f}')

ppo_model = ActorCritic(OBS_DIM, K_STACK, 6, HS, dropout=0.05).to(DEVICE)
ppo_model.load_from_bc(bc_model)
print(f'ActorCritic initialized from BC. params: {sum(p.numel() for p in ppo_model.parameters()):,}')


In [ ]:
progress['stage']='ppo_training'; save_progress()

if OVERCOOKED_INSTALLED:
    try:
        import numpy as np
        np.Inf = np.inf
        np.bool = bool
        np.int = int
        np.float = float
        from overcooked_ai_py.mdp.overcooked_env import OvercookedEnv
        from overcooked_ai_py.mdp.overcooked_mdp import OvercookedGridworld
        from overcooked_ai_py.agents.agent import GreedyHumanModel, RandomAgent, StayAgent
        from overcooked_ai_py.mdp.actions import Action
        # Monkeypatch Action.sample for NumPy 2.0+ compatibility
        Action.sample = lambda action_probs: Action.ALL_ACTIONS[np.random.choice(len(Action.ALL_ACTIONS), p=action_probs)]
        PPO_ENV_OK = True
        print('overcooked_ai_py environment ready')
    except Exception as e:
        PPO_ENV_OK = False
        print(f'overcooked_ai_py import failed: {e}. PPO phase skipped.')
else:
    PPO_ENV_OK = False
    print('overcooked not installed. Skipping PPO phase.')

progress['ppo_env_ok'] = PPO_ENV_OK; save_progress()


In [ ]:
if PPO_ENV_OK:
    # Exclude counter_circuit because GreedyHumanModel does not support tomato recipes
    LAYOUTS = ['cramped_room','coordination_ring','forced_coordination','asymmetric_advantages']
    HORIZON = 250

    def build_env(layout, seed=42):
        mdp = OvercookedGridworld.from_layout_name(layout)
        return OvercookedEnv.from_mdp(mdp, horizon=HORIZON, info_level=0)

    def featurize(env, state, ai):
        return env.featurize_state_mdp(state)[ai].astype(np.float32)

    class EpsGreedy:
        def __init__(self, base, eps, seed=42):
            self.base=base; self.eps=eps; self.rng=np.random.default_rng(seed)
        def action(self, s):
            if self.rng.random()<self.eps:
                return Action.INDEX_TO_ACTION[int(self.rng.integers(0,6))], {}
            return self.base.action(s)
        def reset(self): pass
        def set_agent_index(self, i): self.base.set_agent_index(i)
        def set_mdp(self, m): self.base.set_mdp(m)

    def make_partner(ptype, env, seed=42):
        if ptype=='stay': return StayAgent()
        if ptype=='random': return RandomAgent(all_actions=True)
        base = GreedyHumanModel(env.mlam)
        if ptype=='greedy': return base
        eps = {'greedy10':0.1,'greedy25':0.25,'greedy40':0.40}.get(ptype,0.25)
        return EpsGreedy(base, eps, seed)

    PTYPES = ['stay','random','greedy','greedy10','greedy25','greedy40']
    PSCORES = {p: [] for p in PTYPES}

    def sample_partner_cole(eps=0.2, beta=1.0):
        avg = np.array([np.mean(PSCORES[p]) if PSCORES[p] else .5 for p in PTYPES])
        rng_s = avg.max()-avg.min()
        n = (avg-avg.min())/rng_s if rng_s>0 else np.full(len(avg),.5)
        x=-beta*n; x=x-x.max(); sm=np.exp(x)/np.exp(x).sum()
        probs=(1-eps)*sm+eps/len(PTYPES); probs/=probs.sum()
        return np.random.choice(PTYPES, p=probs)

    # Precompute and cache all environment/partner combinations to avoid massive overhead
    print('Precomputing planning graphs for all environments and partners...')
    t_start_cache = time.time()
    ENV_CACHE = {}
    PARTNER_CACHE = {}
    for lay in LAYOUTS:
        print(f'  Precomputing {lay}...')
        env = build_env(lay)
        ENV_CACHE[lay] = env
        PARTNER_CACHE[lay] = {}
        for pt in PTYPES:
            PARTNER_CACHE[lay][pt] = make_partner(pt, env)
    print(f'Precomputation done! Took {time.time() - t_start_cache:.1f}s')

    # PPO hyperparameters
    GAMMA=0.99; LAM=0.95; CLIP=0.2; LR=3e-4
    ENT_S=0.05; ENT_E=0.01; ENT_D=100000; VC=0.5; MG=0.5
    PPO_EP=4; MB=512; RS=2048; TOTAL=150_000

    opt_ppo = optim.Adam(ppo_model.parameters(), lr=LR)

    def make_stack(dq, k=4):
        return np.concatenate(list(dq)).astype(np.float32)

    gstep=0; ep_rets=[]; ep_soups=[]; ppo_hist=[]
    best_soups=-1e9

    # Rollout buffers
    rb_st=np.zeros((RS,K_STACK*OBS_DIM),np.float32)
    rb_ai=np.zeros(RS,np.int64); rb_pa=np.zeros(RS,np.int64)
    rb_ac=np.zeros(RS,np.int64); rb_rw=np.zeros(RS,np.float32)
    rb_dn=np.zeros(RS,np.float32); rb_vl=np.zeros(RS,np.float32)
    rb_lp=np.zeros(RS,np.float32)

    # Init episode
    cur_l=np.random.choice(LAYOUTS); cur_pt=sample_partner_cole(); cur_role=np.random.randint(0,2)
    env=ENV_CACHE[cur_l]; partner=PARTNER_CACHE[cur_l][cur_pt]
    env.reset(regen_mdp=False); state=env.state
    dq=deque([np.zeros(OBS_DIM,np.float32)]*K_STACK,maxlen=K_STACK)
    dq.append((featurize(env,state,cur_role)-NORM_MEAN)/NORM_STD)
    prev_a=0; ep_r=0.; ep_s=0.; t0_ppo=time.time()

    print(f'PPO start. Total steps: {TOTAL:,}')
    while gstep<TOTAL:
        ppo_model.eval()
        for si in range(RS):
            st_np=make_stack(dq,K_STACK)
            with torch.no_grad():
                logits,val=ppo_model(
                    torch.from_numpy(st_np).unsqueeze(0).to(DEVICE),
                    torch.tensor([cur_role],dtype=torch.long,device=DEVICE),
                    torch.tensor([prev_a],dtype=torch.long,device=DEVICE))
                dist=torch.distributions.Categorical(logits=logits)
                ac=dist.sample(); lp=dist.log_prob(ac)
            ac_i=int(ac.item())
            ego_a=Action.INDEX_TO_ACTION[ac_i]
            partner.set_agent_index(1-cur_role); partner.set_mdp(env.mdp)
            pa_a,_=partner.action(state)
            ja=[ego_a,pa_a] if cur_role==0 else [pa_a,ego_a]
            ns,rew,done,_=env.step(ja)
            ep_r+=float(rew); ep_s+=float(rew)/20.
            rb_st[si]=st_np; rb_ai[si]=cur_role; rb_pa[si]=prev_a
            rb_ac[si]=ac_i; rb_rw[si]=float(rew); rb_dn[si]=float(done)
            rb_vl[si]=float(val.item()); rb_lp[si]=float(lp.item())
            prev_a=ac_i; state=ns
            dq.append((featurize(env,state,cur_role)-NORM_MEAN)/NORM_STD)
            if done:
                ep_rets.append(ep_r); ep_soups.append(ep_s)
                PSCORES[cur_pt].append(ep_s)
                if len(PSCORES[cur_pt])>20: PSCORES[cur_pt]=PSCORES[cur_pt][-20:]
                cur_l=np.random.choice(LAYOUTS); cur_pt=sample_partner_cole()
                cur_role=np.random.randint(0,2)
                env=ENV_CACHE[cur_l]; partner=PARTNER_CACHE[cur_l][cur_pt]
                env.reset(regen_mdp=False); state=env.state
                dq=deque([np.zeros(OBS_DIM,np.float32)]*K_STACK,maxlen=K_STACK)
                dq.append((featurize(env,state,cur_role)-NORM_MEAN)/NORM_STD)
                prev_a=0; ep_r=0.; ep_s=0.

        # Bootstrap
        with torch.no_grad():
            _,lv=ppo_model(torch.from_numpy(make_stack(dq,K_STACK)).unsqueeze(0).to(DEVICE),
                           torch.tensor([cur_role],dtype=torch.long,device=DEVICE),
                           torch.tensor([prev_a],dtype=torch.long,device=DEVICE))
        lv_f=float(lv.item()); ld=bool(rb_dn[RS-1])

        # GAE
        adv=np.zeros(RS,np.float32); g=0.
        for t in reversed(range(RS)):
            nnt=1.-ld if t==RS-1 else 1.-rb_dn[t+1]
            nv=lv_f if t==RS-1 else rb_vl[t+1]
            d_=rb_rw[t]+GAMMA*nv*nnt-rb_vl[t]
            g=d_+GAMMA*LAM*nnt*g; adv[t]=g
        rets=adv+rb_vl
        adv=(adv-adv.mean())/(adv.std()+1e-8)

        # PPO updates
        ppo_model.train()
        pgs=[]; vs=[]; ents=[]
        ec=ENT_S+(ENT_E-ENT_S)*min(gstep/ENT_D,1.)
        for _ in range(PPO_EP):
            for st in range(0,RS,MB):
                idx=np.random.permutation(RS)[st:st+MB]
                logits,vals=ppo_model(
                    torch.from_numpy(rb_st[idx]).to(DEVICE),
                    torch.from_numpy(rb_ai[idx]).to(DEVICE),
                    torch.from_numpy(rb_pa[idx]).to(DEVICE))
                d=torch.distributions.Categorical(logits=logits)
                nlp=d.log_prob(torch.from_numpy(rb_ac[idx]).to(DEVICE))
                ent=d.entropy().mean()
                mb_a=torch.from_numpy(adv[idx]).to(DEVICE)
                mb_r=torch.from_numpy(rets[idx]).to(DEVICE)
                olp=torch.from_numpy(rb_lp[idx]).to(DEVICE)
                rat=torch.exp(nlp-olp)
                pg=torch.max(-mb_a*rat,-mb_a*torch.clamp(rat,1-CLIP,1+CLIP)).mean()
                vl_=F.mse_loss(vals,mb_r)
                loss=pg+VC*vl_-ec*ent
                opt_ppo.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(ppo_model.parameters(),MG); opt_ppo.step()
                pgs.append(pg.item()); vs.append(vl_.item()); ents.append(ent.item())

        gstep+=RS
        if len(ep_rets)>=5:
            ms=np.mean(ep_soups[-20:]); zs=(np.array(ep_soups[-20:])<.5).mean()
            et=time.time()-t0_ppo
            r={'step':gstep,'mean_soups':round(float(ms),4),'zero_soup_rate':round(float(zs),4),
               'pg_loss':round(float(np.mean(pgs)),6),'entropy':round(float(np.mean(ents)),4),'elapsed_s':round(et,1)}
            ppo_hist.append(r)
            print(f'  Step {gstep:7d}: soups={ms:.2f} zero={zs:.2f} ent={ec:.4f} ({et:.0f}s)')
            if ms>best_soups:
                best_soups=ms
                torch.save({'step':gstep,'model_state_dict':ppo_model.state_dict(),'mean_soups':ms,'zero_soup_rate':zs,
                            'obs_dim':OBS_DIM,'k_stack':K_STACK,'hidden_sizes':list(HS)}, OUTPUT_DIR/'best_checkpoint_by_soups.pt')
                print(f'    ✓ new best soups={best_soups:.3f}')
                progress['ppo_best_soups']=float(best_soups); save_progress()
            with open(OUTPUT_DIR/'option_b_ppo_training.csv','w',newline='') as f:
                w=csv.DictWriter(f,fieldnames=list(r.keys())); w.writeheader(); w.writerows(ppo_hist)

    print(f'PPO done. best_soups={best_soups:.3f}')
    progress['artifacts'].append('best_checkpoint_by_soups.pt'); save_progress()
else:
    print('PPO skipped (no overcooked env). Using BC model as final.')


In [ ]:
progress['stage']='export'; save_progress()

# Choose best model
ppo_ckpt = OUTPUT_DIR/'best_checkpoint_by_soups.pt'
if ppo_ckpt.exists():
    ck = torch.load(ppo_ckpt, map_location=DEVICE, weights_only=False)
    ppo_model.load_state_dict(ck['model_state_dict']); ppo_model.eval()
    export_model = ppo_model
    print(f'Using PPO model. step={ck["step"]}, soups={ck["mean_soups"]:.3f}')
    params = {}
    for n,p in ppo_model.actor_encoder.named_parameters(): params[f'actor_encoder.{n}']=p.detach().cpu().numpy()
    for n,p in ppo_model.actor_head.named_parameters(): params[f'actor_head.{n}']=p.detach().cpu().numpy()
else:
    print('Using BC model (PPO not run)')
    bc_model.eval()
    params = {}
    for n,p in bc_model.encoder.named_parameters(): params[f'actor_encoder.{n}']=p.detach().cpu().numpy()
    for n,p in bc_model.actor_head.named_parameters(): params[f'actor_head.{n}']=p.detach().cpu().numpy()

np.savez_compressed(OUTPUT_DIR/'final_policy.npz', **params)
print('Saved final_policy.npz. Keys:', list(params.keys())[:5],'...')

cfg = {'option':'B','model_type':'fcp_ppo_mlp' if ppo_ckpt.exists() else 'bc_warmstart_mlp',
       'obs_dim':OBS_DIM,'k_stack':K_STACK,'num_actions':6,'hidden_sizes':list(HS),
       'normalization_path':'artifacts/shared/normalization.json',
       'agent_index_encoding':'one_hot','previous_action':True,'obs_stack_k':K_STACK}
(OUTPUT_DIR/'final_policy_config.json').write_text(json.dumps(cfg,indent=2))
print('Saved final_policy_config.json')
progress['artifacts'].extend(['final_policy.npz','final_policy_config.json']); save_progress()


In [ ]:
exported = np.load(OUTPUT_DIR/'final_policy.npz')

def ln_np(x, w, b, eps=1e-5):
    m=x.mean(-1,keepdims=True); v=x.var(-1,keepdims=True)
    return (x-m)/np.sqrt(v+eps)*w+b

def np_forward(st, ai, pa):
    ah=np.zeros(2,np.float32); ah[ai]=1.
    ph=np.zeros(6,np.float32); ph[pa]=1.
    x=np.concatenate([st.flatten(),ah,ph])
    li=0
    num_layers = len(HS)
    for i in range(num_layers):
        is_last = (i == num_layers - 1)
        W=exported[f'actor_encoder.net.{li}.weight']; b=exported[f'actor_encoder.net.{li}.bias']
        x=x@W.T+b; li+=1
        if not is_last:
            x=ln_np(x,exported[f'actor_encoder.net.{li}.weight'],exported[f'actor_encoder.net.{li}.bias']); li+=1
            x=np.maximum(0.,x); li+=1; li+=1  # ReLU + Dropout skip
    return x@exported['actor_head.weight'].T+exported['actor_head.bias']

max_err=0; match_=0; N=10
for _ in range(N):
    ts=np.random.randn(K_STACK*OBS_DIM).astype(np.float32)
    tai=np.random.randint(0,2); tpa=np.random.randint(0,6)
    with torch.no_grad():
        if PPO_ENV_OK and ppo_ckpt.exists():
            pt_l=ppo_model.actor_logits(torch.from_numpy(ts).unsqueeze(0).to(DEVICE),
                                        torch.tensor([tai],device=DEVICE),
                                        torch.tensor([tpa],device=DEVICE)).cpu().numpy()[0]
        else:
            pt_l=bc_model(torch.from_numpy(ts).unsqueeze(0).to(DEVICE),
                          torch.tensor([tai],device=DEVICE),
                          torch.tensor([tpa],device=DEVICE)).cpu().numpy()[0]
    np_l=np_forward(ts,tai,tpa)
    err=np.abs(pt_l-np_l).max(); max_err=max(max_err,err)
    if np.argmax(pt_l)==np.argmax(np_l): match_+=1

print(f'Parity: max_err={max_err:.2e}, action_match={match_}/{N}')
ok=max_err<=1e-4 and match_==N
(OUTPUT_DIR/'parity_check.json').write_text(json.dumps({'max_abs_error':float(max_err),'match_rate':match_/N,'parity_ok':ok},indent=2))
progress['parity_ok']=ok; save_progress()
print('Parity OK:', ok)


In [ ]:
eval_results = []
if PPO_ENV_OK:
    progress['stage']='evaluation'; save_progress()
    
    # Define helper inside the cell to make it completely self-contained
    def make_stack(dq, k=4):
        return np.concatenate(list(dq)).astype(np.float32)
        
    EVAL_L=['cramped_room','coordination_ring','forced_coordination']
    EVAL_P=['stay','random','greedy']
    EVAL_S=[42,43,44]; EVAL_R=[0,1]
    eval_model = ppo_model if ppo_ckpt.exists() else bc_model

    for lay in EVAL_L:
        for pt in EVAL_P:
            for sd in EVAL_S:
                for role in EVAL_R:
                    try:
                        ev=build_env(lay,sd); par=make_partner(pt,ev,sd)
                        ev.reset(regen_mdp=False); st=ev.state
                        dq2=deque([np.zeros(OBS_DIM,np.float32)]*K_STACK,maxlen=K_STACK)
                        dq2.append((featurize(ev,st,role)-NORM_MEAN)/NORM_STD)
                        pa2=0; ep_r2=0.; ep_s2=0.; done2=False; ts2=0; fst=None; lst=None
                        while not done2:
                            with torch.no_grad():
                                if hasattr(eval_model,'actor_logits'):
                                    lg=eval_model.actor_logits(torch.from_numpy(make_stack(dq2,K_STACK)).unsqueeze(0).to(DEVICE),torch.tensor([role],device=DEVICE),torch.tensor([pa2],device=DEVICE))
                                else:
                                    lg=eval_model(torch.from_numpy(make_stack(dq2,K_STACK)).unsqueeze(0).to(DEVICE),torch.tensor([role],device=DEVICE),torch.tensor([pa2],device=DEVICE))
                                ac2=int(lg.argmax(-1).item())
                            ea=Action.INDEX_TO_ACTION[ac2]
                            par.set_agent_index(1-role); par.set_mdp(ev.mdp)
                            pa_a2,_=par.action(st)
                            ja2=[ea,pa_a2] if role==0 else [pa_a2,ea]
                            nst,rw,done2,_=ev.step(ja2)
                            ep_r2+=float(rw)
                            if rw>0:
                                ep_s2+=float(rw)/20.
                                if fst is None: fst=ts2
                                lst=ts2
                            pa2=ac2; st=nst; dq2.append((featurize(ev,st,role)-NORM_MEAN)/NORM_STD); ts2+=1
                        sc=10000*ep_s2+(10*(HORIZON-lst)+HORIZON-fst if ep_s2>0 else 0)
                        eval_results.append({'layout':lay,'partner':pt,'seed':sd,'role':role,
                                             'soups':ep_s2,'sparse_return':ep_r2,'official_score':sc})
                        print(f'  {lay}+{pt} s={sd} r={role}: soups={ep_s2:.1f}')
                    except Exception as e: print(f'  Error {lay}+{pt}: {e}')

    if eval_results:
        with open(OUTPUT_DIR/'option_b_evaluation.csv','w',newline='') as f:
            w=csv.DictWriter(f,fieldnames=list(eval_results[0].keys())); w.writeheader(); w.writerows(eval_results)
        ms=np.mean([r['soups'] for r in eval_results]); zs=np.mean([r['soups']<.5 for r in eval_results])
        print(f'Eval: mean_soups={ms:.3f}, zero_soup_rate={zs:.3f}')
        (OUTPUT_DIR/'eval_summary.json').write_text(json.dumps({'mean_soups':float(ms),'zero_soup_rate':float(zs),'n_eps':len(eval_results)},indent=2))
        progress['eval_summary']={'mean_soups':float(ms),'zero_soup_rate':float(zs)}
    progress['artifacts'].extend(['option_b_evaluation.csv','eval_summary.json']); save_progress()
else:
    print('Skipping evaluation (no overcooked env)')


In [ ]:
try:
    # Partner pool manifest
    pm={'partners':[{'id':'stay'},{'id':'random'},{'id':'greedy'},{'id':'greedy10','noise':0.1},{'id':'greedy25','noise':0.25},{'id':'greedy40','noise':0.4}]}
    (OUTPUT_DIR/'partner_pool_manifest.json').write_text(json.dumps(pm,indent=2))

    # Final artifact list
    print('Output files:')
    for f in sorted(OUTPUT_DIR.iterdir()):
        if f.is_file(): print(f'  {f.name}: {f.stat().st_size/1024:.1f} KB')

    progress['status']='complete'; progress['stage']='done'
    progress['artifacts'].append('partner_pool_manifest.json'); save_progress()
    print('=== All done! ===')
except Exception as e:
    progress['status']='failed'; progress['error']=repr(e); save_progress()
    import traceback; traceback.print_exc()
